# A2A Agent Evaluation

This notebook provides a structured way to evaluate A2A agents using the Vertex AI Evaluation Service. It uses the `a2a-sdk` for agent interaction and the Vertex AI SDK (`google-cloud-aiplatform`) for running evaluation metrics. The notebook uses Application Default Credentials(ADC) for interacting with Google Cloud Services.

If you are not already logged in, run `gcloud auth application-default login` to create the default credentials.

In [1]:
# Install required SDKs
%pip install a2a-sdk google-cloud-aiplatform[evaluation] pandas httpx

Looking in indexes: https://us-python.pkg.dev/artifact-foundry-prod/ah-3p-staging-python/simple/

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setup the Agent URL and GCP Project to use for evaluations

In [3]:
# --- Config ---
# Replace with the URL of your deployed agent
AGENT_URL = 'http://localhost:10999'

# Replace with your GCP project ID
PROJECT_ID = 'kcthota-eval-sandbox-764418'
LOCATION = 'us-central1'

import vertexai


vertexai.init(project=PROJECT_ID, location=LOCATION)

In [5]:
import json
import re

from typing import Any
from urllib.parse import urlparse

import httpx
import pandas as pd
import vertexai

from a2a.client import ClientConfig, ClientFactory, create_text_message_object

### Define any custom metrics

In [6]:
from vertexai.preview.evaluation import PointwiseMetric


text_similarity = PointwiseMetric(
    metric='text_similarity',
    metric_prompt_template="""
            You are an expert in evaluating the quality of text generated by AI models. Your task is to compare a generated text with a reference text and determine how similar they are in meaning.

            You should consider the following factors:
            1. **Semantic Similarity**: Do the two texts convey the same meaning?
            2. **Content Overlap**: How many of the key points from the reference text are present in the generated text?
            3. **Relevance**: Is the generated text relevant to the topic of the reference text?
            4. **Completeness**: Does the generated text cover all the important aspects of the reference text?

            Provide a score between 1 and 5, where:
            - 5 means the texts are identical in meaning
            - 1 means the texts have no relation in meaning

            User Prompt:
            {prompt}

            Generated Text:
            {response}

            Reference Text:
            {reference}
        """,
)

### A2A Response Parsing Helpers

These utility functions make it easier to extract specific parts of the agent's response for evaluation.


In [7]:
def get_node_text(node: Any) -> str:
    """Extracts and concatenates all text parts from a Status or Artifact."""
    if not node:
        return ''
    text_parts = []
    # Status nodes often have a message attribute with parts
    # Artifact nodes have parts or contents
    parts = getattr(node, 'parts', None) or getattr(node, 'contents', None)
    if not parts and hasattr(node, 'message'):
        parts = getattr(node.message, 'parts', [])

    for part in parts or []:
        actual_part = getattr(part, 'root', part)
        if hasattr(actual_part, 'text'):
            text_parts.append(actual_part.text)
        elif isinstance(actual_part, dict) and 'text' in actual_part:
            text_parts.append(actual_part['text'])
    return '\n'.join(text_parts).strip()


def get_status_message(task: Any) -> str:
    """Retrieves the status message text from the task."""
    return get_node_text(task.status)


def get_artifact_by_index(task: Any, index: int) -> Any | None:
    """Retrieves the artifact at the specified index."""
    if task.artifacts and 0 <= index < len(task.artifacts):
        return task.artifacts[index]
    return None


def get_artifacts_by_name(task: Any, name_regex: str) -> list[Any]:
    """Retrieves artifacts whose name matches the regex pattern."""
    if not task.artifacts:
        return []
    return [
        a
        for a in task.artifacts
        if hasattr(a, 'name') and a.name and re.search(name_regex, a.name)
    ]


def get_aggregated_artifact_parts(artifact: Any) -> str:
    """Aggregates all text parts from an artifact into a single string."""
    return get_node_text(artifact)

In [8]:
# --- Agent Invocation & Data Extraction ---

TRACE_KEY = (
    'github.com/a2aproject/a2a-samples/extensions/traceability/v1/traceability'
)


async def eval_agent(test_case: dict) -> pd.DataFrame:
    """Evaluates the agent with a given test case and returns a result DataFrame.

    Args:
        test_case: A dictionary containing the prompt, reference, and other evaluation data.

    Returns:
        A pandas DataFrame with evaluation results.
    """
    parsed = urlparse(AGENT_URL)
    agent_url = f'{parsed.scheme}://{parsed.netloc}'
    relative_card_path = (
        parsed.path if parsed.path and parsed.path != '/' else None
    )

    # Enable traceability extension for trajectory retrieval
    headers = {
        'x-a2a-extensions': 'https://github.com/a2aproject/a2a-samples/extensions/traceability/v1'
    }

    client = await ClientFactory.connect(
        agent_url,
        relative_card_path=relative_card_path,
        client_config=ClientConfig(
            httpx_client=httpx.AsyncClient(headers=headers, timeout=60)
        ),
    )

    message = create_text_message_object(content=test_case['prompt'])
    stream = client.send_message(message)
    events = [event async for event in stream]
    task = events[-1][0]

    # --- Dynamic Data Extraction based on 'select' ---
    # --- customize this function to return the response from agent returned task object
    select = test_case.get('select', {})
    from_source = select.get('from')
    where = select.get('where', {})

    response_text = ''
    if from_source == 'status':
        response_text = get_status_message(task)
    elif from_source == 'artifacts':
        index = where.get('index')
        name_match = where.get('name_match')

        if index is not None:
            artifact = get_artifact_by_index(task, index)
            response_text = (
                get_aggregated_artifact_parts(artifact) if artifact else ''
            )
        elif name_match:
            artifacts = get_artifacts_by_name(task, name_match)
            response_text = '\n'.join(
                [get_aggregated_artifact_parts(a) for a in artifacts]
            )
        else:
            # Aggregate all artifacts
            response_text = '\n'.join(
                [
                    get_aggregated_artifact_parts(a)
                    for a in (task.artifacts or [])
                ]
            )
    else:
        # Default: Aggregate all artifacts
        response_text = '\n'.join(
            [get_aggregated_artifact_parts(a) for a in (task.artifacts or [])]
        )

    # 2. Extract trajectory (from metadata trace)
    predicted_trajectory = []
    if hasattr(task, 'metadata') and task.metadata:
        trace_obj = task.metadata.get(TRACE_KEY, task.metadata.get('trace', {}))
        if isinstance(trace_obj, dict) and 'steps' in trace_obj:
            predicted_trajectory.extend(
                [
                    {
                        'tool_name': step.get('name'),
                        'tool_input': step.get('parameters'),
                    }
                    for step in trace_obj['steps']
                    if step.get('step_type') == 'tool_call'
                ]
            )

    return pd.DataFrame(
        [
            {
                'prompt': test_case['prompt'],
                'response': response_text.strip(),
                'reference': test_case.get('reference', ''),
                'predicted_trajectory': json.dumps(
                    predicted_trajectory, ensure_ascii=False
                ),
                'reference_trajectory': json.dumps(
                    test_case.get('reference_trajectory', []),
                    ensure_ascii=False,
                ),
            }
        ]
    )

### Define Test Data
Test cases can be organized into Test sets. This allow evaluating common metrics on similar test cases and reporting the results as one experiment.

In [9]:
# --- Define Test Data ---
# Test sets allow you to group test cases and share common configurations.
# 'experiment_name' is used as the experiment name in Vertex AI.
# 'common_metrics' provide defaults for test cases that don't specify their own.
test_sets = [
    {
        'experiment_name': 'currency-conversion-v1',
        'common_metrics': [
            'coherence',
            'rouge_1',
            'trajectory_exact_match',
            text_similarity,
        ],
        'active': True,
        'test_cases': [
            {
                'name': 'conversion-test-1',
                'prompt': 'how much is 10 USD in CAD?',
                'select': {
                    'from': 'artifacts',
                    'where': {'name_match': 'conversion_result'},
                },
                'reference': '10 USD is approximately 13.66 CAD',
                'reference_trajectory': [
                    {
                        'tool_name': 'get_latest_rates',
                        'tool_input': {'symbols': 'CAD', 'base': 'USD'},
                    }
                ],
            },
            {
                'name': 'conversion-test-2',
                'prompt': 'how much is 10 USD in INR?',
                'select': {
                    'from': 'artifacts',
                    'where': {'name_match': 'conversion_result'},
                },
                'reference': '10 USD is approximately 920 INR',
                'reference_trajectory': [
                    {
                        'tool_name': 'get_latest_rates',
                        'tool_input': {'symbols': 'INR', 'base': 'USD'},
                    }
                ],
            },
        ],
    },
    {
        'experiment_name': 'supported-currencies-v1',
        'common_metrics': [
            'coherence',
            'trajectory_exact_match',
            text_similarity,
        ],
        'active': True,
        'test_cases': [
            {
                'name': 'supported-1',
                'prompt': 'find the list of supported currencies for EU',
                'select': {'from': 'artifacts', 'where': {'index': 0}},
                'reference': 'Supported EU currencies are: EUR,CZK,DKK,HUF,PLN,RON,SEK',
                'reference_trajectory': [
                    {'tool_name': 'get_available_currencies', 'tool_input': {}}
                ],
            }
        ],
    },
    {
        'experiment_name': 'clarification-v1',
        'common_metrics': ['coherence', 'fluency', text_similarity],
        'active': True,
        'test_cases': [
            {
                'name': 'usd-clarification',
                'prompt': 'how much is 10 USD',
                'select': {'from': 'status'},
                'reference': 'Please provide the currency to convert to.',
            }
        ],
    },
]

### Run the evaluation and report the results

In [11]:
# --- Evaluation Execution ---
from vertexai.preview.evaluation import EvalTask


# We will store results for inspection
all_eval_results = {}

for test_set in test_sets:
    experiment_name = test_set.get('experiment_name', 'agent-eval')
    if not test_set.get('active', True):
        print(f'Skipping Test Set {experiment_name}. (Reason: active=false)')
        continue

    common_metrics = test_set.get('common_metrics', ['rouge_1'])

    print(f'\n📊 Starting Test Set: {experiment_name}')

    # 1. Collect results for all test cases in this set
    set_results_list = []
    for tc in test_set.get('test_cases', []):
        test_name = tc['name']

        if not tc.get('active', True):
            print(f'Skipping Test Case {test_name}. (Reason: active=false)')
            continue

        print(f'🚀 Running Agent for: {tc["name"]}')
        case_df = await eval_agent(tc)
        set_results_list.append(case_df)

    if not set_results_list:
        print(f'⚠️ No test cases found in set {experiment_name}')
        continue

    # 2. Aggregate into a single DataFrame for the whole set
    set_df = pd.concat(set_results_list, ignore_index=True)

    # 3. Define and run ONE EvalTask for the entire set
    print(f'🔍 Running Evaluation for set: {experiment_name}')
    eval_task = EvalTask(
        dataset=set_df, metrics=common_metrics, experiment=experiment_name
    )

    # 4. Execute evaluation
    eval_result = eval_task.evaluate()
    all_eval_results[experiment_name] = eval_result

    # 5. Show summary and detailed results for the set
    print(f'✅ Results for Set {experiment_name}:')
    display(eval_result.metrics_table)


📊 Starting Test Set: currency-conversion-v1
🚀 Running Agent for: conversion-test-1
🚀 Running Agent for: conversion-test-2
🔍 Running Evaluation for set: currency-conversion-v1


Associating projects/944809475955/locations/us-central1/metadataStores/default/contexts/currency-conversion-v1-d6f4ad44-914b-4e71-9cba-59b59e014a72 to Experiment: currency-conversion-v1


Computing metrics with a total of 8 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 8/8 [00:09<00:00,  1.19s/it]

2 errors encountered during evaluation. Continue to compute summary metrics for the rest of the dataset.
Error encountered for metric trajectory_exact_match at dataset index 0: Error: 400 List of found errors:	1.Field: trajectory_exact_match_input.instances[0].predicted_trajectory.tool_calls; Message: Required field is not set.	 [field_violations {
  field: "trajectory_exact_match_input.instances[0].predicted_trajectory.tool_calls"
  description: "Required field is not set."
}
, detail: "[ORIGINAL ERROR] generic::invalid_argument: com.google.cloud.ai.platform.common.errors.AiPlatformException: code=INVALID_ARGUMENT, message=List of found errors:\t1.Field: trajectory_exact_match_input.instances[0].predicted_trajectory.tool_calls; Message: Required field is not set.\t, cause=null [google.rpc.error_details_ext] { code: 3 message: \"List of found errors:\\t1.Field: trajectory_exact_match_input.instances[0].predicted_trajectory.tool_calls; Message: Required field is not set.\\t\" details { 

...

1 errors encountered during evaluation. Continue to compute summary metrics for the rest of the dataset.
Error encountered for metric trajectory_exact_match at dataset index 0: Error: 400 List of found errors:	1.Field: trajectory_exact_match_input.instances[0].predicted_trajectory.tool_calls; Message: Required field is not set.	 [field_violations {
  field: "trajectory_exact_match_input.instances[0].predicted_trajectory.tool_calls"
  description: "Required field is not set."
}
, detail: "[ORIGINAL ERROR] generic::invalid_argument: com.google.cloud.ai.platform.common.errors.AiPlatformException: code=INVALID_ARGUMENT, message=List of found errors:\t1.Field: trajectory_exact_match_input.instances[0].predicted_trajectory.tool_calls; Message: Required field is not set.\t, cause=null [google.rpc.error_details_ext] { code: 3 message: \"List of found errors:\\t1.Field: trajectory_exact_match_input.instances[0].predicted_trajectory.tool_calls; Message: Required field is not set.\\t\" details { 

✅ Results for Set supported-currencies-v1:


,prompt,response,reference,predicted_trajectory,reference_trajectory,coherence/explanation,coherence/score,trajectory_exact_match/score,text_similarity/explanation,text_similarity/score
0,find the list of supported currencies for EU,The supported currencies available are:\n- AUD...,"Supported EU currencies are: EUR,CZK,DKK,HUF,P...",[],"[{""tool_name"": ""get_available_currencies"", ""to...",The response demonstrates strong logical flow ...,4.0,None,The generated text includes all the currencies...,3.0



📊 Starting Test Set: clarification-v1
🚀 Running Agent for: usd-clarification


CancelledError: 